# 02 — Regressão logística: versão mínima

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flavioluizseixas/aprendizado-de-maquina-para-saude/blob/main/notebooks/02_aprendizado_supervisionado_vers%C3%A3o_minima.ipynb)

Execute no Colab ou em Jupyter com `pandas` e `scikit-learn` instalados. O carregamento requer internet.

## Fonte e licença

[CDC Diabetes Health Indicators — UCI 891](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators). Consulte a fonte para os códigos das variáveis e os termos de uso.

**Alvo:** 0 = sem diabetes; 1 = pré-diabetes/diabetes. Para simplificar, os códigos ordinais de `Age`, `GenHlth`, `Education` e `Income` permanecem numéricos.

Material com finalidade exclusivamente educacional.

## 1. Dados e split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

dados = pd.read_csv("https://archive.ics.uci.edu/static/public/891/data.csv", index_col="ID")
X, y = dados.drop(columns="Diabetes_binary"), dados["Diabetes_binary"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## 2. Padronização e treinamento

O pipeline aprende a média, o desvio-padrão e os coeficientes somente no treino. `predict` usa limiar 0,5.

In [ ]:
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]

## 3. Desempenho no teste

No relatório, `recall` da classe 1 é a sensibilidade; da classe 0, a especificidade. Observe também precisão, F1 e acurácia.

In [ ]:
print(classification_report(y_test, y_pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

## 4. Coeficientes da regressão logística

Coeficiente positivo aumenta a log-odds estimada da classe 1; negativo a reduz, mantendo os demais atributos constantes.

Os coeficientes usam a **escala padronizada**: nas variáveis contínuas, uma unidade corresponde a um desvio-padrão do treino. Nas binárias, a mudança de 0 para 1 não equivale a um desvio-padrão. São coeficientes de um modelo regularizado; não representam efeitos causais.

In [ ]:
pd.Series(modelo[-1].coef_[0], index=X.columns, name="Coeficiente").sort_values().round(3)